# Resultados Laura MCTS vs Mario

Este notebook resume las 50 partidas jugadas entre `Laura MCTS` y `Group mario`, alternando quien inicia. El objetivo es analizar resultado, efecto del primer jugador y costo computacional.

**Configuracion:**

- `Laura MCTS`: agente MCTS con `num_simulations = 100`.
- `Group mario`: agente tactico/heuristico basado en evaluacion lineal de ventanas.
- Partidas: 50.
- Alternancia: Laura inicia 25 partidas y Mario inicia 25 partidas.


## Carga de datos

El archivo `laura_vs_mario_50_results.csv` tiene dos filas por partida: una desde la perspectiva de Laura y otra desde la perspectiva de Mario. Para evitar duplicar conteos, los resumenes por partida usan solo las filas donde `focal_side == 'agent_1'`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / 'laura_vs_mario_50_results.csv'
FIG_DIR = PROJECT_ROOT / 'figuras_entrega'

df = pd.read_csv(CSV_PATH)
games = df[df['focal_side'] == 'agent_1'].copy().sort_values('game_id')

print('Filas del CSV:', len(df))
print('Partidas reales:', games['game_id'].nunique())
print('Tiempo total medido (min):', round(games['total_game_time'].sum() / 60, 2))

games.head()


## Resumen global

| Resultado | Partidas | Porcentaje |
| --- | ---: | ---: |
| Gana Laura | 25 | 50% |
| Gana Mario | 25 | 50% |
| Empate | 0 | 0% |

El resultado global queda completamente balanceado: 25 victorias para Laura y 25 para Mario.

In [ ]:
winner_summary = (
    games['winner_agent']
    .value_counts()
    .rename_axis('winner_agent')
    .reset_index(name='partidas')
)
winner_summary['porcentaje'] = winner_summary['partidas'] / len(games)
winner_summary


## Resultado segun quien inicia

| Quien inicia | Partidas | Gana Laura | Gana Mario | Empates | Win rate Laura |
| --- | ---: | ---: | ---: | ---: | ---: |
| Laura inicia | 25 | 0 | 25 | 0 | 0% |
| Mario inicia | 25 | 25 | 0 | 0 | 100% |

El patron fue extremo: gano siempre el jugador que iba segundo. Cuando Laura inicio, gano Mario; cuando Mario inicio, gano Laura.

In [ ]:
by_starter = (
    games.groupby(['starter', 'winner_agent'])
    .size()
    .unstack(fill_value=0)
)
by_starter


## Grafica 1: resultados alternando inicio

La primera grafica muestra el resultado global y luego separa el resultado segun quien inicia.

![Resultados Laura vs Mario](figuras_entrega/14_laura_vs_mario_resultados.png)

## Tiempos y duracion

| Metrica | Valor |
| --- | ---: |
| Tiempo total medido | 8.50 min |
| Tiempo promedio por partida | 10.20 s |
| Movimientos promedio por partida | 35.00 |

Mario es practicamente instantaneo. El tiempo de cada partida esta dominado por las decisiones de Laura MCTS.

In [ ]:
timing_summary = pd.DataFrame({
    'metrica': [
        'tiempo_total_min',
        'tiempo_promedio_partida_s',
        'movimientos_promedio',
    ],
    'valor': [
        games['total_game_time'].sum() / 60,
        games['total_game_time'].mean(),
        games['num_moves'].mean(),
    ],
})
timing_summary


## Grafica 2: costo computacional

La segunda grafica compara el tiempo promedio por jugada de cada agente y muestra la duracion de cada partida.

![Tiempos Laura vs Mario](figuras_entrega/15_laura_vs_mario_tiempos.png)

## Lectura final

El empate global 25-25 es enganoso si se mira sin separar por orden de juego. La variable decisiva fue quien empieza: en esta configuracion siempre gano el segundo jugador. Esto sugiere que el enfrentamiento Laura vs Mario tiene una dinamica muy estable y dependiente del orden inicial.

## Regenerar graficas

Esta celda es opcional. Solo hace falta ejecutarla si se modifica el CSV o si se quieren volver a exportar las imagenes.

In [ ]:
REGENERAR_GRAFICAS = False

if REGENERAR_GRAFICAS:
    import runpy
    runpy.run_path('generar_graficas_laura_vs_mario.py')
